In [ ]:
import pandas as pd
import numpy as np
import krippendorff
import sys
import csv
import re
from itertools import chain
import pyreadstat

In [ ]:
actor_df = pd.read_csv('actor_annotations.csv')

# remove first two rows
actor_df = actor_df.iloc[2:]
print(actor_df.shape)
actor_df.head()

In [ ]:
coder_dutch_extra = pd.read_csv('coder_dutch_extra.csv')
# remove first two rows
coder_dutch_extra = coder_dutch_extra.iloc[2:]
print(coder_dutch_extra.shape)
coder_dutch_extra.head()

In [ ]:
actor_df = pd.concat([actor_df, coder_dutch_extra], ignore_index=True)
actor_df.shape

In [ ]:
# get the column names that have ActorNamem ActorFunction or ActorPP in them
cols = []
for col in actor_df.columns:
    if 'ActorName' in col or 'ActorFunction' in col or 'ActorPP' in col:
        cols.append(col)
print(cols)

In [ ]:
cols.append('RecordedDate')
cols.append('M1. Coder')
cols.append('M2. ArticleID')
cols.append('M3. ArticleTitle')

print(len(cols))
print(cols)

actor_df = actor_df[cols]

# rename columns to remove spaces
actor_df.columns = [col.replace(' ', '_') for col in actor_df.columns]
actor_df.head()

In [ ]:
# make M2._ArticleID an integer
actor_df['M2._ArticleID'] = actor_df['M2._ArticleID'].astype(int)

articles_to_remove = [14014326, 11000981, 14030203, 12000108, 13005098, 14000379]

# drop if M2._ArticleID is in articles_to_remove
actor_df = actor_df[~actor_df['M2._ArticleID'].isin(articles_to_remove)]
print(actor_df.shape)

In [ ]:
actor_df.head()

In [ ]:
# are there multiple M2._ArticleID per M1._Codeer
actor_df.groupby(['M1._Coder', 'M2._ArticleID']).size().reset_index(name='counts').sort_values(by='counts', ascending=False).head()

In [ ]:
# deduplicate based on M1._Coder and M2._ArticleID, keeping the last occurrence based on RecordedDate
actor_df = actor_df.sort_values(by=['M2._ArticleID', 'M1._Coder', 'RecordedDate'])
actor_df = actor_df.drop_duplicates(subset=['M1._Coder', 'M2._ArticleID'], keep='last')
actor_df.head()

In [ ]:
actor_df.groupby(['M1._Coder', 'M2._ArticleID']).size().reset_index(name='counts').sort_values(by='counts', ascending=False).head()

In [ ]:
# drop RecordedDate column
actor_df = actor_df.drop(columns=['RecordedDate'])

# rename M1._Coder to Coder and M2._ArticleID to ArticleID
actor_df = actor_df.rename(columns={'M1._Coder': 'Coder', 'M2._ArticleID': 'ArticleID', 'M3._ArticleTitle': 'ArticleTitle'})
actor_df.head()

In [ ]:
# Transform wide actor columns (1_V3._ActorName ... 30_V5._ActorPP) into actor-level rows


# find actor prefixes like "1_", "2_", ... by scanning column names
actor_pattern = re.compile(r'^(\d+)_V3\._ActorName$')
actor_nums = []
for col in actor_df.columns:
    m = actor_pattern.match(col)
    if m:
        actor_nums.append(m.group(1))

actor_nums = sorted(set(actor_nums), key=lambda x: int(x))  # keep numeric order
len(actor_nums), actor_nums[:5]  # quick check

In [ ]:
# build a list of (wide->long) DataFrame fragments, then concat
rows = []
for n in actor_nums:
    name_col = f"{n}_V3._ActorName"
    func_col = f"{n}_V4._ActorFunction"
    pp_col   = f"{n}_V5._ActorPP"
    # some files might miss columns for high n, so guard
    if name_col not in actor_df.columns:
        continue
    tmp = actor_df[[ 'Coder', 'ArticleID', 'ArticleTitle', name_col, func_col, pp_col ]].copy()
    tmp = tmp.rename(columns={
        name_col: 'ActorName',
        func_col: 'ActorFunction',
        pp_col: 'ActorPP'
    })
    tmp['ActorNumber'] = int(n)
    rows.append(tmp)

In [ ]:
# concat all actor-numbered blocks
actor_long = pd.concat(rows, ignore_index=True, sort=False)
actor_long.head()

In [ ]:
# optionally drop rows where ActorName is blank/NaN (no actor in that slot)
actor_long['ActorName'] = actor_long['ActorName'].astype(str).str.strip()
actor_long = actor_long[actor_long['ActorName'].notna() & (actor_long['ActorName'] != '')].reset_index(drop=True)

# reorder columns
actor_long = actor_long[['Coder', 'ArticleID', 'ArticleTitle', 'ActorNumber', 'ActorName', 'ActorFunction', 'ActorPP']]

# show result
print(actor_long.shape)
actor_long.head()

In [ ]:
# sort by ArticleID, ActorNumber and Coder
actor_long = actor_long.sort_values(by=['ArticleID', 'ActorNumber', 'Coder']).reset_index(drop=True)
actor_long.head()   

In [ ]:
actor_long.ActorName.value_counts(dropna=False)
# drop if ActorName is 'nan' or empty string
actor_long = actor_long[(actor_long['ActorName']!= 'nan') & (actor_long['ActorName'] != '')].reset_index(drop=True)
print(actor_long.shape)
actor_long.head()

In [ ]:
actor_long.isnull().sum()

In [ ]:
actor_long.Coder.value_counts()
# drop if coder is researcher 
actor_long = actor_long[actor_long['Coder'] != 'researcher'].reset_index(drop=True)
print(actor_long.shape)

In [ ]:
actor_long.head()
# drop ArticleTitle column
actor_long = actor_long.drop(columns=['ArticleTitle'])
actor_long.head()

In [ ]:
# make a new column to concat ArticleID and ActorNumber
actor_long['Unique_Actor_ID'] = actor_long['ArticleID'].astype(str) + '_' + actor_long['ActorNumber'].astype(str)
actor_long.head()

In [ ]:
# make the coder integer
coder_mapping = {name: idx+1 for idx, name in enumerate(sorted(actor_long['Coder'].unique()))}
coder_mapping

In [ ]:
actor_long['Coder'] = actor_long['Coder'].map(coder_mapping)
actor_long.head()

In [ ]:
len(actor_long)

actor_long[actor_long['Coder']==1].Unique_Actor_ID.nunique(), actor_long[actor_long['Coder']==2].Unique_Actor_ID.nunique()

In [ ]:
actors_coder_turkish = actor_long[actor_long['Coder']==1]
actors_coder_turkish.ActorFunction.value_counts(dropna=False)

In [ ]:
actors_coder_dutch = actor_long[actor_long['Coder']==2]
actors_coder_dutch.ActorFunction.value_counts(dropna=False)

In [ ]:
actors_coder_turkish.ArticleID.nunique()

In [ ]:
actors_coder_dutch.ArticleID.nunique()

In [ ]:
# see which ArticleIDs are in actors_coder_turkish but not in actors_coder_dutch
ali_article_ids = set(actors_coder_turkish['ArticleID'].unique())
coder_dutch_article_ids = set(actors_coder_dutch['ArticleID'].unique())
ali_not_in_coder_dutch = ali_article_ids - coder_dutch_article_ids
ali_not_in_coder_dutch

In [ ]:
actors_coder_dutch.head()

In [ ]:
# add a row from coder_dutch where article id is ali_not_in_coder_dutch

new_row = {'Coder': 2,
           'ArticleID': list(ali_not_in_coder_dutch)[0],
           'ActorNumber': 0,
            'ActorName': 'N/A',
            'ActorFunction': 'N/A',
            'ActorPP': 'N/A',
            'Unique_Actor_ID': f"{list(ali_not_in_coder_dutch)[0]}_0"
           }

actors_coder_dutch = pd.concat([actors_coder_dutch, pd.DataFrame([new_row])], ignore_index=True)
actors_coder_dutch.tail()

In [ ]:
actor_long.head()

# str.strip and upper actor
actor_long['ActorName'] = actor_long['ActorName'].str.strip().str.upper()

In [ ]:
# save actor_long to csv
actor_long.to_csv('actor_long.csv', index=False, sep=';', encoding='utf-8', quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
# Save actor_long to SPSS (.sav) safely using pyreadstat
# This cell will try to install pyreadstat if it's missing. Re-run the cell after installation if needed.

# make a copy to avoid mutating the working df
df_to_save = actor_long.copy()

# Ensure column names are SPSS-friendly (no leading/trailing spaces)
df_to_save.columns = [str(c).strip() for c in df_to_save.columns]

# Prepare numeric columns: ArticleID and ActorNumber.
# SPSS requires numeric variables to be numeric (int/float). We keep NaN as system-missing by using float if any missing exist.
for col in ['ArticleID', 'ActorNumber']:
    if col in df_to_save.columns:
        df_to_save[col] = pd.to_numeric(df_to_save[col], errors='coerce')
        if df_to_save[col].isnull().any():
            # leave as float64 so NaN -> system-missing in SPSS
            df_to_save[col] = df_to_save[col].astype('float64')
        else:
            # safe to convert to integer numpy type
            df_to_save[col] = df_to_save[col].astype('int64')

# Ensure text columns are plain Python strings or NaN (pyreadstat will map NaN to system-missing)
for col in ['Coder', 'ActorName', 'ActorFunction', 'ActorPP', 'Unique_Actor_ID']:
    if col in df_to_save.columns:
        df_to_save[col] = df_to_save[col].where(df_to_save[col].notna(), None)
        # convert non-missing values to str (preserve content)
        df_to_save.loc[df_to_save[col].notna(), col] = df_to_save.loc[df_to_save[col].notna(), col].astype(str)

# Optional: set variable/column labels (helpful in SPSS GUI)
variable_labels = {
    'Coder': 'Coder name',
    'ArticleID': 'Original Article ID',
    'ActorNumber': 'Actor slot number',
    'Unique_Actor_ID': 'Unique Actor ID (ArticleID_ActorNumber)',
    'ActorName': 'Name of the actor',
    'ActorFunction': 'Function/role of the actor',
    'ActorPP': 'Public/political party or affiliation (PP)'
}

out_path = 'actor_long.sav'
print('Writing', out_path, '...')

# pyreadstat uses `column_labels` for variable/column labels (not `variable_labels`)
try:
    pyreadstat.write_sav(df_to_save, out_path, file_label='Actor-level coding', column_labels=variable_labels, variable_value_labels=None)
    print('Finished writing', out_path)
except TypeError as e:
    # Fallback: older/newer pyreadstat versions might have different signatures. Try without labels.
    print('write_sav call with column_labels failed:', e)
    print('Retrying write without column labels...')
    pyreadstat.write_sav(df_to_save, out_path, file_label='Actor-level coding', variable_value_labels=None)
    print('Finished writing', out_path, ' (without column labels)')


In [ ]:
# calculate the number of actors per article per coder
actor_counts = actor_long.groupby(['ArticleID', 'Coder']).size().reset_index(name='NumActors')
actor_counts.head()

In [ ]:
actors_coder_turkish = actor_long[actor_long['Coder']==1]
actors_coder_dutch = actor_long[actor_long['Coder']==2]
ali_article_ids = set(actors_coder_turkish['ArticleID'].unique())
coder_dutch_article_ids = set(actors_coder_dutch['ArticleID'].unique())
ali_not_in_coder_dutch = sorted(list(ali_article_ids - coder_dutch_article_ids))

print('ArticleIDs in ali_not_in_coder_dutch (count):', len(ali_not_in_coder_dutch))

# build rows to append: for each missing ArticleID, add (ArticleID, Coder=2, NumActors=0) if not present
rows_to_add = []
for aid in ali_not_in_coder_dutch:
    exists = ((actor_counts['ArticleID'] == aid) & (actor_counts['Coder'] == 2)).any()
    if not exists:
        rows_to_add.append({'ArticleID': aid, 'Coder': 2, 'NumActors': 0})

if rows_to_add:
    actor_counts = pd.concat([actor_counts, pd.DataFrame(rows_to_add)], ignore_index=True, sort=False)
    # keep consistent dtypes
    actor_counts['ArticleID'] = pd.to_numeric(actor_counts['ArticleID'], errors='coerce')
    actor_counts['Coder'] = pd.to_numeric(actor_counts['Coder'], errors='coerce').astype('int64')
    actor_counts['NumActors'] = pd.to_numeric(actor_counts['NumActors'], errors='coerce').astype('int64')
    actor_counts = actor_counts.sort_values(by=['ArticleID', 'Coder']).reset_index(drop=True)

print('Updated actor_counts shape:', actor_counts.shape)
actor_counts.head(20)


In [ ]:
actor_counts[actor_counts['ArticleID'].isin(ali_not_in_coder_dutch)]

In [ ]:
actor_counts[actor_counts['NumActors']==0]

In [ ]:
# calculate Krippendorff's alpha
matrix = actor_counts.pivot(index="Coder", columns="ArticleID", values="NumActors")
reliability_data = matrix.to_numpy()
alpha_value_nr_actors = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="interval")
print("Krippendorff's alpha for number of actors per article:", alpha_value_nr_actors)


In [ ]:
# save this to spss
out_path = 'actor_counts.sav'
pyreadstat.write_sav(actor_counts, out_path, variable_value_labels=None)

In [ ]:
# calculate the number of actors per article per coder
actor_counts_percat = actor_long.groupby(['ArticleID', 'Coder', 'ActorFunction']).size().reset_index(name='NumActors')
actor_counts_percat.head()

In [ ]:
actor_counts_percat.ActorFunction.value_counts()

In [ ]:
# map actor_function values to a b c d
actor_function_mapping = {
    'Government and politics (includes international, national, regional politicians, government/state officials, members of law enforcement)': 'a',
    'Professionals and experts (expets outside government/state organizations, media and business representatives, researchers, academics, celebrities)': 'b',
    'Civil society organizations (non-governmental organizations, non-profit organizations, trade unions, social movements, interest groups)': 'c',
    'Citizens/members of the public (ordinary citizens, residents, people speaking from personal experience rather than professional expertise such as residents affected by policies, immigrants sharing personal stories, eyewitnesses to events, community members expressing individual opinions; this category also includes collective public opinion if quoted/paraphrased)': 'd'
}
actor_counts_percat['ActorFunction'] = actor_counts_percat['ActorFunction'].map(actor_function_mapping)
actor_counts_percat.head()
actor_counts_percat.ActorFunction.value_counts()

In [ ]:
# transpose the df so that every function type is a column with number of actors as values
actor_counts_percat_pivot = actor_counts_percat.pivot_table(index=['ArticleID', 'Coder'], columns='ActorFunction', values='NumActors', fill_value=0).reset_index()
actor_counts_percat_pivot.columns.name = None  # remove the aggregation name
actor_counts_percat_pivot = actor_counts_percat_pivot.reset_index(drop=True)
actor_counts_percat_pivot.head()

In [ ]:
# make it all integer
actor_counts_percat_pivot['a'] = actor_counts_percat_pivot['a'].astype(int)
actor_counts_percat_pivot['b'] = actor_counts_percat_pivot['b'].astype(int)
actor_counts_percat_pivot['c'] = actor_counts_percat_pivot['c'].astype(int)
actor_counts_percat_pivot['d'] = actor_counts_percat_pivot['d'].astype(int)
actor_counts_percat_pivot.head()

In [ ]:
def compute_alpha_per_category(df, categories, level="interval"):
    results = {}
    for cat in categories:
        # pivot to coder x article matrix
        mat = df.pivot(index="Coder", columns="ArticleID", values=cat)
        reliability_data = mat.to_numpy()
        try:
            a_val = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement=level)
        except Exception as e:
            a_val = np.nan
            print(f"⚠️ Category {cat}: could not compute alpha ({e})")
        results[cat] = a_val
    return results

categories = ["a", "b", "c", "d"]
alphas = compute_alpha_per_category(actor_counts_percat_pivot, categories)
print(alphas)

In [ ]:
# save this to spss
out_path = 'actor_counts_perfunctie.sav'
pyreadstat.write_sav(actor_counts_percat_pivot, out_path, variable_value_labels=None)

In [ ]:
# get the sum of actors coded by each coder per category
nr_actors_per_category = actor_counts_percat_pivot.groupby(['Coder']).sum().reset_index()
nr_actors_per_category.head()

# Calculate agreement with corrected actor names

In [ ]:
# read sav file
df, meta = pyreadstat.read_sav('actor_reliability_transformed.sav')
df.head()

In [ ]:
pd.crosstab(df.v1_binary, df.v2_binary)

In [ ]:
# calculate classification report
from sklearn.metrics import classification_report
report = classification_report(df['v1_binary'], df['v2_binary'], zero_division=0)
print(report)

In [ ]:
# calculate function for where v1_binary and v2_binary are both 1
df_both_1 = df[(df['v1_binary']==1) & (df['v2_binary']==1)]
data = df_both_1[['Funcion_cats.1', 'Funcion_cats.2']].to_numpy().T
alpha_func = krippendorff.alpha(reliability_data=data, level_of_measurement='nominal')
print("Krippendorff's alpha for Actor Function (both binary=1):", alpha_func)